In [1]:
import requests
import json
import psycopg2
import pandas as pd

# Configuración de la base de datos PostgreSQL


db_host = "localhost"  # o la dirección IP de tu servidor PostgreSQL
db_name = "cybersecurity_logs"
db_user = "postgres"
db_password = "postgres"
db_port = 5432  # Puerto predeterminado de PostgreSQL
tabla_nombre = "cves"  # Table name for storing CVEs


def conectar_a_db():
    try:
        conn = psycopg2.connect(
            host=db_host,
            database=db_name,
            user=db_user,
            password=db_password,
            port=db_port
        )
        print("Conexión a la base de datos exitosa")
        return conn
    except psycopg2.Error as e:
        print(f"Error al conectar a la base de datos: {e}")
        return None

def crear_tabla(conn):
    try:
        cursor = conn.cursor()
        sql = f"""
            CREATE TABLE IF NOT EXISTS {tabla_nombre} (
                id TEXT PRIMARY KEY,
                source_identifier TEXT,
                published TIMESTAMP,
                last_modified TIMESTAMP,
                vuln_status TEXT,
                descriptions JSONB,
                metrics JSONB,
                weaknesses JSONB,
                configurations JSONB,
                cve_references JSONB  -- Columna renombrada para evitar conflicto con palabra reservada
            );
        """
        cursor.execute(sql)
        conn.commit()
        print(f"Tabla '{tabla_nombre}' creada o ya existe.")
        cursor.close()
    except psycopg2.Error as e:
        print(f"Error al crear la tabla: {e}")
        if conn:
            conn.rollback()

def insertar_cves(conn, cves_data):
    try:
        cursor = conn.cursor()
        for cve_entry in cves_data:
            cve = cve_entry.get("cve")
            if cve:
                try:
                    # Convertir diccionarios a cadenas JSON antes de insertar
                    cve['descriptions'] = json.dumps(cve['descriptions']) if cve.get('descriptions') else None
                    cve['metrics'] = json.dumps(cve['metrics']) if cve.get('metrics') else None
                    cve['weaknesses'] = json.dumps(cve['weaknesses']) if cve.get('weaknesses') else None
                    cve['configurations'] = json.dumps(cve['configurations']) if cve.get('configurations') else None
                    cve['cve_references'] = json.dumps(cve['references']) if cve.get('references') else None  # Usar el nombre de columna renombrado

                    cursor.execute(
                        f"""
                        INSERT INTO {tabla_nombre} (
                            id, source_identifier, published, last_modified, vuln_status, 
                            descriptions, metrics, weaknesses, configurations, cve_references  -- Usar el nombre de columna renombrado
                        ) VALUES (
                            %(id)s, %(sourceIdentifier)s, %(published)s, %(lastModified)s, %(vulnStatus)s,
                            %(descriptions)s::jsonb, %(metrics)s::jsonb, %(weaknesses)s::jsonb, %(configurations)s::jsonb, %(cve_references)s::jsonb  -- Usar el nombre de columna renombrado
                        )
                        """,
                        cve
                    )
                    conn.commit()
                    print(f"CVE {cve['id']} insertado.")
                except psycopg2.Error as e:
                    conn.rollback()
                    print(f"Error al insertar CVE {cve['id']}: {e}")
        cursor.close()
    except Exception as e:
        print(f"Error general al insertar CVEs: {e}")
        if conn:
            conn.rollback()

def main():
    conn = conectar_a_db()
    if not conn:
        return

    crear_tabla(conn)

    url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        if data and "vulnerabilities" in data:
            cves_data = data["vulnerabilities"]
            insertar_cves(conn, cves_data)
        else:
            print("No se encontraron vulnerabilidades en la respuesta de la API.")

    except requests.exceptions.RequestException as e:
        print(f"Error al llamar a la API: {e}")
    except json.JSONDecodeError as e:
        print(f"Error al decodificar JSON: {e}")
    except Exception as e:
        print(f"Error inesperado: {e}")

    if conn:
        conn.close()
        print("Conexión a la base de datos cerrada.")

if __name__ == "__main__":
    main()

Conexión a la base de datos exitosa
Tabla 'cves' creada o ya existe.
CVE CVE-1999-0095 insertado.
CVE CVE-1999-0082 insertado.
CVE CVE-1999-1471 insertado.
CVE CVE-1999-1122 insertado.
CVE CVE-1999-1467 insertado.
CVE CVE-1999-1506 insertado.
CVE CVE-1999-0084 insertado.
CVE CVE-2000-0388 insertado.
CVE CVE-1999-0209 insertado.
CVE CVE-1999-1198 insertado.
CVE CVE-1999-1391 insertado.
CVE CVE-1999-1392 insertado.
CVE CVE-1999-1057 insertado.
CVE CVE-1999-1554 insertado.
CVE CVE-1999-1197 insertado.
CVE CVE-1999-1115 insertado.
CVE CVE-1999-1258 insertado.
CVE CVE-1999-1438 insertado.
CVE CVE-1999-1211 insertado.
CVE CVE-1999-1212 insertado.
CVE CVE-1999-1194 insertado.
CVE CVE-1999-1193 insertado.
CVE CVE-1999-1123 insertado.
CVE CVE-1999-1034 insertado.
CVE CVE-1999-1415 insertado.
CVE CVE-1999-1090 insertado.
CVE CVE-1999-0498 insertado.
CVE CVE-1999-1468 insertado.
CVE CVE-1999-0167 insertado.
CVE CVE-1999-1493 insertado.
CVE CVE-1999-1032 insertado.
CVE CVE-1999-1059 insertado.
CVE